<a href="https://colab.research.google.com/github/KotteBhavani/Major_Project/blob/main/Major_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import mutual_info_classif
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_curve, auc
import joblib
import warnings
warnings.filterwarnings("ignore")

In [ ]:
from google.colab import files
uploaded = files.upload()

TypeError: 'NoneType' object is not subscriptable

In [ ]:
import zipfile
import os

for file in uploaded.keys():
    with zipfile.ZipFile(file, 'r') as zip_ref:
        zip_ref.extractall('/content')

os.listdir('/content')

In [ ]:

# read the CSV files from /content
testing_set   = pd.read_csv('/content/UNSW_NB15_testing-set.csv')
training_set  = pd.read_csv('/content/UNSW_NB15_training-set.csv')
LIST_EVENTS   = pd.read_csv('/content/UNSW-NB15_LIST_EVENTS.csv')
NB15_1        = pd.read_csv('/content/UNSW-NB15_1.csv')
NB15_2        = pd.read_csv('/content/UNSW-NB15_2.csv')
NB15_3        = pd.read_csv('/content/UNSW-NB15_3.csv')
NB15_4        = pd.read_csv('/content/UNSW-NB15_4.csv')

# this file needs a special encoding
NB15_features = pd.read_csv('/content/NUSW-NB15_features.csv', encoding='cp1252')

# check one file to confirm it loaded correctly
print(training_set.head())

In [ ]:
unique_attacks = LIST_EVENTS['Attack category'].dropna().unique()
print(unique_attacks)

In [ ]:
NB15_1.head()

In [ ]:
NB15_2.head()

In [ ]:
NB15_3.head()

In [ ]:
NB15_4.head()

In [ ]:
NB15_features

In [ ]:
NB15_1.columns = NB15_features['Name']
NB15_2.columns = NB15_features['Name']
NB15_3.columns = NB15_features['Name']
NB15_4.columns = NB15_features['Name']

In [ ]:
train_df = pd.concat([NB15_1, NB15_2, NB15_3, NB15_4], ignore_index=True)

In [ ]:
# Categorical columns to encode later
cat_cols = ['proto', 'state', 'service']

# Numerical columns
num_cols = train_df.select_dtypes(include=['int64', 'float64']).columns

# Fill missing numerical values with median
train_df[num_cols] = train_df[num_cols].fillna(train_df[num_cols].median())

# Fill missing categorical values with mode
for col in cat_cols:
    train_df[col] = train_df[col].fillna(train_df[col].mode()[0])

# Convert categorical columns to string type
for col in cat_cols:
    train_df[col] = train_df[col].astype(str)


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Normal vs Attack
sns.countplot(x=train_df['Label'])
plt.title("Normal vs Attack Traffic")
plt.show()

# Attack category distribution
train_df['attack_cat'].value_counts().plot(kind='bar')
plt.title("Attack Category Distribution")
plt.show()


In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
for col in cat_cols:
    train_df[col] = le.fit_transform(train_df[col])


In [ ]:
X = train_df.drop(columns=['Label', 'attack_cat'])
y = train_df['Label']

# Drop remaining non-numeric columns if any
X = X.select_dtypes(include=['int64','float64'])


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)


In [ ]:
# ==============================
# 🔹 MUTUAL INFORMATION FEATURE SELECTION (FIXED)
# ==============================

import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_selection import mutual_info_classif
from sklearn.model_selection import train_test_split

# Recreate X (before scaling) for MI
X_mi = train_df.drop(columns=['Label', 'attack_cat'])
y_mi = train_df['Label']

# Handle non-numeric columns
cat_cols_mi = X_mi.select_dtypes(include='object').columns
num_cols_mi = X_mi.select_dtypes(include=['int64', 'float64']).columns

# Impute numeric columns
num_imputer = SimpleImputer(strategy='mean')
X_num = pd.DataFrame(
    num_imputer.fit_transform(X_mi[num_cols_mi]),
    columns=num_cols_mi
)

# Impute + Encode categorical columns
cat_imputer = SimpleImputer(strategy='most_frequent')
X_cat = pd.DataFrame(
    cat_imputer.fit_transform(X_mi[cat_cols_mi]),
    columns=cat_cols_mi
)

le = LabelEncoder()
for col in X_cat.columns:
    X_cat[col] = le.fit_transform(X_cat[col].astype(str))

# Combine numeric + categorical
X_mi_final = pd.concat([X_num, X_cat], axis=1)

# ==============================
# 🔹 SPEED FIX: SAMPLE DATA FOR MI
# ==============================

SAMPLE_SIZE = min(20000, len(X_mi_final))  # safe for Colab
X_sample = X_mi_final.sample(n=SAMPLE_SIZE, random_state=42)
y_sample = y_mi.loc[X_sample.index]

# Compute Mutual Information (FAST)
mi_scores = mutual_info_classif(
    X_sample,
    y_sample,
    n_neighbors=3,
    n_jobs=-1
)

mi_series = pd.Series(mi_scores, index=X_mi_final.columns)\
               .sort_values(ascending=False)

print("Top 15 Features by Mutual Information:")
print(mi_series.head(15))

# Select top-k features
TOP_K = 20
top_features = mi_series.head(TOP_K).index.tolist()

# ==============================
# 🔹 TRAIN XGBOOST WITH TOP FEATURES
# ==============================

from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report

X_selected = X_mi_final[top_features]

X_train_fs, X_test_fs, y_train_fs, y_test_fs = train_test_split(
    X_selected,
    y_mi,
    test_size=0.2,
    random_state=42,
    stratify=y_mi
)

scaler = StandardScaler()
X_train_fs = scaler.fit_transform(X_train_fs)
X_test_fs = scaler.transform(X_test_fs)

xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='logloss'
)

xgb_model.fit(X_train_fs, y_train_fs)

y_pred = xgb_model.predict(X_test_fs)

print("\n✅ XGBoost Accuracy:", accuracy_score(y_test_fs, y_pred))
print(classification_report(y_test_fs, y_pred))


In [ ]:
import queue
import threading
import time
import pandas as pd
import matplotlib.pyplot as plt
import gradio as gr


In [ ]:
# Simulated Kafka topic
kafka_topic = queue.Queue(maxsize=100)


In [ ]:
def kafka_producer(sample_df):
    """
    Simulates Kafka Producer:
    Takes input data → model → sends prediction
    """
    sample_scaled = scaler.transform(sample_df)
    prediction = int(xgb_model.predict(sample_scaled)[0])

    message = {
        "timestamp": time.time(),
        "prediction": prediction
    }

    kafka_topic.put(message)
    print("📤 Produced:", message)


In [ ]:
# Store last N predictions for visualization
prediction_buffer = []

def kafka_consumer():
    """
    Simulates Kafka Consumer:
    Continuously listens and stores predictions
    """
    while True:
        message = kafka_topic.get()
        prediction_buffer.append(message)

        # Keep only last 50 records
        if len(prediction_buffer) > 50:
            prediction_buffer.pop(0)

        print("📥 Consumed:", message)


In [ ]:
consumer_thread = threading.Thread(target=kafka_consumer, daemon=True)
consumer_thread.start()


In [ ]:
def simulate_live_data():
    """
    Sends random samples from test set
    """
    sample = X_test_fs[:1]  # pick any row
    sample_df = pd.DataFrame(sample, columns=top_features)

    kafka_producer(sample_df)


In [ ]:
def visualize_predictions():
    if len(prediction_buffer) == 0:
        return None

    preds = [msg["prediction"] for msg in prediction_buffer]

    plt.figure(figsize=(6,4))
    plt.plot(preds, marker='o')
    plt.title("Real-Time Intrusion Detection")
    plt.xlabel("Time")
    plt.ylabel("Prediction (0 = Normal, 1 = Attack)")
    plt.ylim(-0.2, 1.2)
    return plt


In [ ]:
with gr.Blocks() as demo:
    gr.Markdown("## 🚨 Real-Time Intrusion Detection System")

    btn = gr.Button("Send New Network Packet")
    plot = gr.Plot()

    btn.click(fn=simulate_live_data)
    demo.load(fn=visualize_predictions, outputs=plot, every=2)

demo.launch()


In [ ]:
# ==============================
# 🔹 FINAL IDS DASHBOARD WITH RED ATTACK ALERTS
# ==============================

import time
import random
import pandas as pd
import plotly.express as px
import gradio as gr

# ------------------------------
# 🔹 CONFIG
# ------------------------------

ATTACK_TYPES = ["DoS", "Probe", "R2L", "U2R", "PortScan"]
TOTAL_SAMPLES = 20


# ------------------------------
# 🔹 DATA GENERATION (SIMULATION)
# ------------------------------

def generate_stream_data():
    data = []
    attack_positions = random.sample(range(TOTAL_SAMPLES), 6)  # force attacks

    for i in range(TOTAL_SAMPLES):
        if i in attack_positions:
            data.append({
                "Sample": i,
                "Prediction": "Attack",
                "Attack_Type": random.choice(ATTACK_TYPES),
                "Status": "❌"
            })
        else:
            data.append({
                "Sample": i,
                "Prediction": "Normal",
                "Attack_Type": "—",
                "Status": "✅"
            })
    return data


# ------------------------------
# 🔹 BAR GRAPH
# ------------------------------

def plot_stream(df):
    fig = px.histogram(
        df,
        x="Prediction",
        color="Prediction",
        title="🔴 Intrusion Detection Live Stream",
        text_auto=True
    )
    fig.update_layout(bargap=0.4)
    return fig


# ------------------------------
# 🔹 ALERT PANEL (HTML)
# ------------------------------

def build_alert_panel(df):
    html = ""
    for _, row in df.iterrows():
        if row["Prediction"] == "Attack":
            html += f"""
            <div style="
                background-color:#ff4d4d;
                color:white;
                padding:10px;
                margin:5px;
                border-radius:8px;
                font-weight:bold;">
                ❌ Sample {row['Sample']} — ATTACK DETECTED ({row['Attack_Type']})
            </div>
            """
        else:
            html += f"""
            <div style="
                background-color:#4CAF50;
                color:white;
                padding:10px;
                margin:5px;
                border-radius:8px;">
                ✅ Sample {row['Sample']} — Normal Traffic
            </div>
            """
    return html


# ------------------------------
# 🔹 STREAM FUNCTION
# ------------------------------

def stream_predictions():
    stream_df = pd.DataFrame(columns=["Sample", "Prediction", "Attack_Type", "Status"])
    stream_data = generate_stream_data()

    for row in stream_data:
        stream_df.loc[len(stream_df)] = row
        yield (
            plot_stream(stream_df),
            build_alert_panel(stream_df)
        )
        time.sleep(0.4)


# ------------------------------
# 🔹 GRADIO DASHBOARD
# ------------------------------

with gr.Blocks() as dashboard:
    gr.Markdown("## 🛡️ Intrusion Detection System Dashboard")
    gr.Markdown("### 🔴 Real‑Time Attack Alerts")
    gr.Markdown("❌ **Attack (Red Box)**  ✅ **Normal (Green Box)**")

    start_btn = gr.Button("▶ Start Live Detection")

    output_plot = gr.Plot()
    alert_panel = gr.HTML()

    start_btn.click(
        fn=stream_predictions,
        outputs=[output_plot, alert_panel]
    )

dashboard.launch(share=True)
